# Crop annotation

Pulls one crop at a time from the existing WS-DINO loader, shows the **t1**
global crop as a max-intensity projection, and lets you assign a single class
with a button click. Each label writes the full 3D t1 crop (`.npy`), a MIP
preview (`.png`), and an append-only JSONL record under `OUT_DIR` (see
`data/annotate_io.py`).

Augmentations are **disabled** here so you label clean crops.

**Resume / dedup.** Each session uses a fresh random seed by default, so you
see new crops on resume rather than replaying the old sequence. Crops already
**labeled** or **skipped** are suppressed (never shown twice). **Skip** saves
the crop to `skipped.jsonl` so you can revisit all skips in bulk later (the
*Review skipped* section) — e.g. when they turn out to be a class you hadn't
defined yet.

Run all cells, then click a class button per crop. Use the **predicty** kernel.

In [ ]:
%matplotlib inline
import sys, os, random
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from configs.config import Config
from data.dataset import ZarrCellDataset, MultiZarrDataset
from data.annotate_io import (
    make_uid, save_annotation, save_skip, relabel_skip,
    load_seen, load_skipped, load_suppressed, load_pending_skips,
)


# --- display helpers (mirrored from notebooks/visualize.ipynb) ---
def mip(volume):
    """max projection across z; bright structures stand out."""
    return volume.max(axis=0)


def percentile_limits(values, low=1.0, high=99.0):
    """display limits robust to a few bright outliers."""
    lo, hi = np.percentile(np.asarray(values), [low, high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.min(values)), float(np.max(values))
    if hi <= lo:
        hi = lo + 1e-6
    return float(lo), float(hi)

## Config

Edit `CLASSES` and point at your data. `Config.from_yaml` keeps every other
knob at its default; we force `num_workers = 0` (single-process iteration),
turn off augmentation, and set a per-session seed.

`SESSION_SEED = None` draws a fresh random region of the dataset each session
(recommended — avoids replaying skipped crops and re-reading labeled ones on
resume). Set it to an int if you ever want a reproducible ordering.

In [ ]:
# classes you want to annotate into (single label per crop)
CLASSES = ['class_a', 'class_b', 'class_c']

# where labels + crops are written (gitignored)
OUT_DIR = '../annotations/labels'

# None -> fresh random crops each session; set an int for a reproducible order
SESSION_SEED = None

# build a config: from the training yaml, or inline e.g.
#   cfg = Config(zarr_path='/path/to/experiment.zarr')
cfg = Config.from_yaml('../configs/train.yaml')

# single-process iteration in the notebook (no DataLoader / worker fork)
cfg.num_workers = 0

# per-session seed (the dataset builds its RNG from cfg.seed)
cfg.seed = SESSION_SEED if SESSION_SEED is not None else random.randrange(2**31)

# annotate clean crops: disable every augmentation, including per-crop norm
for aug in (cfg.aug_intensity, cfg.aug_gamma, cfg.aug_noise,
            cfg.aug_blur, cfg.aug_percrop_norm):
    aug.enabled = False

print('classes:', CLASSES)
print('out_dir:', os.path.abspath(OUT_DIR))
print('session seed:', cfg.seed)

## Build the dataset + iterator

In [ ]:
if cfg.zarr_paths:
    ds = MultiZarrDataset(cfg)
else:
    ds = ZarrCellDataset(cfg)

it = iter(ds)

# suppress anything already labeled OR skipped so it is never shown again
suppress = load_suppressed(OUT_DIR)
n_labeled = len(load_seen(OUT_DIR))
n_skipped = len(load_skipped(OUT_DIR))
print(f'resuming: {n_labeled} labeled, {n_skipped} skipped (both suppressed)')


def resolve_zarr_path(ds, dataset_idx):
    """map a sample's dataset_idx back to its source zarr path."""
    if isinstance(ds, MultiZarrDataset):
        return ds.children[int(dataset_idx)].cfg.zarr_path
    return ds.cfg.zarr_path

## Annotate

Click a class to save the current crop and advance. **Unsure** saves with the
label `unsure` (a normal label). **Skip** stashes the crop in `skipped.jsonl`
to revisit later. Both labeled and skipped crops are suppressed from here on.

In [ ]:
out = widgets.Output()
status = widgets.Label()
current = {'sample': None, 'uid': None}
counts = {'labeled': n_labeled, 'skipped': n_skipped}


def _update_status():
    status.value = (f"labeled: {counts['labeled']}   skipped: {counts['skipped']}"
                    f"   |   current: {current['uid']}")


def _draw():
    sample = current['sample']
    crop = sample['globals'][0, 0].numpy()  # (Z, H, W), t1, channel 0
    img = mip(crop)
    lims = percentile_limits(img, 0.5, 99.5)
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img, cmap='gray', vmin=lims[0], vmax=lims[1])
        m = sample['meta']
        ax.set_title(f"ds{m['dataset_idx']}  t1={m['t1']}  z={m['z']}", fontsize=10)
        ax.axis('off')
        plt.show()


def _advance():
    while True:
        sample = next(it)
        uid = make_uid(sample['meta'], int(sample['globals'].shape[-1]))
        if uid not in suppress:
            break
    current['sample'] = sample
    current['uid'] = uid
    _draw()
    _update_status()


def _make_label_handler(label):
    def handler(_btn):
        sample = current['sample']
        zp = resolve_zarr_path(ds, sample['meta']['dataset_idx'])
        rec = save_annotation(OUT_DIR, sample, label, zarr_path=zp)
        suppress.add(rec['uid'])
        counts['labeled'] += 1
        _advance()
    return handler


def _on_skip(_btn):
    sample = current['sample']
    zp = resolve_zarr_path(ds, sample['meta']['dataset_idx'])
    rec = save_skip(OUT_DIR, sample, zarr_path=zp)
    suppress.add(rec['uid'])
    counts['skipped'] += 1
    _advance()


class_buttons = []
for label in CLASSES:
    b = widgets.Button(description=label, button_style='primary')
    b.on_click(_make_label_handler(label))
    class_buttons.append(b)

unsure_btn = widgets.Button(description='Unsure', button_style='warning')
unsure_btn.on_click(_make_label_handler('unsure'))
skip_btn = widgets.Button(description='Skip', button_style='')
skip_btn.on_click(_on_skip)

controls = widgets.HBox(class_buttons + [unsure_btn, skip_btn])
display(widgets.VBox([status, controls, out]))
_advance()

## Review skipped

Walk every crop you skipped and not yet labeled, loading the saved pixels (no
dataset needed). Assign a class to move it into `index.jsonl`, or **Keep
skipped** to leave it for later. Re-run this cell to refresh the queue after
labeling some — e.g. once you have defined the new class they belong to.

In [ ]:
review_out = widgets.Output()
review_status = widgets.Label()
rstate = {'queue': load_pending_skips(OUT_DIR), 'i': 0, 'rec': None}


def _review_draw():
    rec = rstate['rec']
    crop = np.load(rec['crop_path'])  # (1, Z, H, W)
    img = mip(crop[0])
    lims = percentile_limits(img, 0.5, 99.5)
    with review_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img, cmap='gray', vmin=lims[0], vmax=lims[1])
        ax.set_title(rec['uid'], fontsize=9)
        ax.axis('off')
        plt.show()


def _review_advance():
    q = rstate['queue']
    if rstate['i'] >= len(q):
        rstate['rec'] = None
        with review_out:
            clear_output(wait=True)
            print('no more pending skips')
        review_status.value = f'done: reviewed all {len(q)} pending skip(s)'
        return
    rstate['rec'] = q[rstate['i']]
    rstate['i'] += 1
    _review_draw()
    review_status.value = f"pending {rstate['i']}/{len(q)}   uid: {rstate['rec']['uid']}"


def _make_review_handler(label):
    def handler(_btn):
        relabel_skip(OUT_DIR, rstate['rec'], label)
        _review_advance()
    return handler


def _keep_skipped(_btn):
    _review_advance()


review_buttons = []
for label in CLASSES:
    b = widgets.Button(description=label, button_style='primary')
    b.on_click(_make_review_handler(label))
    review_buttons.append(b)
keep_btn = widgets.Button(description='Keep skipped', button_style='')
keep_btn.on_click(_keep_skipped)

review_controls = widgets.HBox(review_buttons + [keep_btn])
display(widgets.VBox([review_status, review_controls, review_out]))
if rstate['queue']:
    _review_advance()
else:
    with review_out:
        print('no pending skips to review')